# Test that PyDAP can download dap responses from DAP4 servers, and decode the information


- Pydap users python's requests library, as well as requests_cache to create session object to stream data. It has a de-serializer for dap4 responses, which was written based NOT on the spec, but on the experience of receiving dap responses from Hyrax.

The test file is located in both Thredds and Hyrax tests servers. These are identical files. The file structure is as follows:


```python
## tree directory inside simply group
pyds.tree()
.SimpleGroup.nc4
├──SimpleGroup
│  ├──Temperature
│  ├──Salinity
│  ├──Y
│  └──X
├──Pressure
├──time_bnds
├──time
└──Z

```
## Warning
It has become recently know to me that the order in which items appear on the representation of the file tree above, is the way in which pydap will de-serialize a dap response. THis is because the DMR parser in pydap creates FIRST Emtpy Groups containers, and later populates them. THis is a simple fix, however, fixing this will create a deserialization problem with Hyrax, given that we found that (by chance) Hyrax serializes this way too! So to bugs working well together.





In [1]:
from pydap.client import open_url, create_session
import numpy as np
import time
import pydap

In [2]:
print("pydap version: ", pydap.__version__)

pydap version:  3.5.8


In [3]:
hyx_url = "dap4://test.opendap.org/opendap/dap4/SimpleGroup.nc4.h5"
tds_url = "dap4://thredds-test.unidata.ucar.edu/thredds/dap4/dev/d4icomp/SimpleGroup.nc4"

In [4]:
session=create_session(use_cache=True, cache_kwargs={'cache_name':'debug'})


# Lets begin with Hyrax first

In [5]:
session.cache.clear()
pyds = open_url(hyx_url, batch=True, session=session)
session.cache.clear()
# Group_vars = 
all_vars = list(pyds.variables()) + list([pyds['SimpleGroup'][var].id for var in pyds["SimpleGroup"].variables()])

array_data = {}
for var in all_vars:
    array_data[var] = pyds[var][:].data

time.sleep(2)

print("================================================================================ \n DAP response from Hyrax Server: \n ", session.cache.urls()[0].replace("%5B","[").replace("%5D","]").replace("%3A",":").replace("%2F","/").replace("%3B",";"), "\n================================================================================")

 DAP response from Hyrax Server: 
  http://test.opendap.org/opendap/dap4/SimpleGroup.nc4.h5.dap?dap4.ce=/SimpleGroup/Salinity[0:1:0][0:1:39][0:1:39];/time[0:1:0];/Pressure[0:1:999];/SimpleGroup/Temperature[0:1:0][0:1:39][0:1:39];/SimpleGroup/X[0:1:39];/time_bnds[0:1:0][0:1:1];/Z[0:1:999];/SimpleGroup/Y[0:1:39]&dap4.checksum=true 


In [6]:
print("================================================================================ \n Now lets look that the data makes sense: \n================================================================================")
for var in array_data:
    array_data[var] = np.asarray(array_data[var])
array_data

 Now lets look that the data makes sense: 


{'time': array([0.5], dtype=float32),
 'Z': array([  -0.,   -1.,   -2.,   -3.,   -4.,   -5.,   -6.,   -7.,   -8.,
          -9.,  -10.,  -11.,  -12.,  -13.,  -14.,  -15.,  -16.,  -17.,
         -18.,  -19.,  -20.,  -21.,  -22.,  -23.,  -24.,  -25.,  -26.,
         -27.,  -28.,  -29.,  -30.,  -31.,  -32.,  -33.,  -34.,  -35.,
         -36.,  -37.,  -38.,  -39.,  -40.,  -41.,  -42.,  -43.,  -44.,
         -45.,  -46.,  -47.,  -48.,  -49.,  -50.,  -51.,  -52.,  -53.,
         -54.,  -55.,  -56.,  -57.,  -58.,  -59.,  -60.,  -61.,  -62.,
         -63.,  -64.,  -65.,  -66.,  -67.,  -68.,  -69.,  -70.,  -71.,
         -72.,  -73.,  -74.,  -75.,  -76.,  -77.,  -78.,  -79.,  -80.,
         -81.,  -82.,  -83.,  -84.,  -85.,  -86.,  -87.,  -88.,  -89.,
         -90.,  -91.,  -92.,  -93.,  -94.,  -95.,  -96.,  -97.,  -98.,
         -99., -100., -101., -102., -103., -104., -105., -106., -107.,
        -108., -109., -110., -111., -112., -113., -114., -115., -116.,
        -117., -118., -119., -120.

## Conclusion: The entire data is downloaded from Hyrax, and after being decoded by Pydap it is correct


# Now the Thredds Data Server

In [7]:
session.cache.clear()
pyds = open_url(tds_url, batch=True, session=session)
session.cache.clear()
# Group_vars = 
all_vars = list(pyds.variables()) + list([pyds['SimpleGroup'][var].id for var in pyds["SimpleGroup"].variables()])

array_data = {}
for var in all_vars:
    array_data[var] = pyds[var][:].data

time.sleep(2)

print("================================================================================ \n DAP response from Thredds Data Server: \n ", session.cache.urls()[0].replace("%5B","[").replace("%5D","]").replace("%3A",":").replace("%2F","/").replace("%3B",";"), "\n================================================================================")

 DAP response from Thredds Data Server: 
  https://thredds-test.unidata.ucar.edu/thredds/dap4/dev/d4icomp/SimpleGroup.nc4.dap?dap4.ce=/SimpleGroup/Salinity[0:1:0][0:1:39][0:1:39];/time[0:1:0];/Pressure[0:1:999];/SimpleGroup/Temperature[0:1:0][0:1:39][0:1:39];/SimpleGroup/X[0:1:39];/time_bnds[0:1:0][0:1:1];/Z[0:1:999];/SimpleGroup/Y[0:1:39]&dap4.checksum=true 


In [8]:
print("================================================================================ \n Now lets look that the data makes sense: \n================================================================================")
for var in array_data:
    array_data[var] = np.asarray(array_data[var])
array_data

 Now lets look that the data makes sense: 


{'Pressure': array([ 1.237000e+03,  1.238000e+03,  1.239000e+03,  1.240000e+03,
         1.241000e+03,  1.242000e+03,  1.243000e+03,  1.244000e+03,
         1.245000e+03,  1.246000e+03,  1.247000e+03,  1.248000e+03,
         1.249000e+03,  1.250000e+03,  1.251000e+03,  1.252000e+03,
         1.253000e+03,  1.254000e+03,  1.255000e+03,  1.256000e+03,
         1.257000e+03,  1.258000e+03,  1.259000e+03,  1.260000e+03,
         1.261000e+03,  1.262000e+03,  1.263000e+03,  1.264000e+03,
         1.265000e+03,  1.266000e+03,  1.267000e+03,  1.268000e+03,
         1.269000e+03,  1.270000e+03,  1.271000e+03,  1.272000e+03,
         1.273000e+03,  1.274000e+03,  1.275000e+03,  1.276000e+03,
         1.277000e+03,  1.278000e+03,  1.279000e+03,  1.280000e+03,
         1.281000e+03,  1.282000e+03,  1.283000e+03,  1.284000e+03,
         1.285000e+03,  1.286000e+03,  1.287000e+03,  1.288000e+03,
         1.289000e+03,  1.290000e+03,  1.291000e+03,  1.292000e+03,
         1.293000e+03,  1.294000e+03

## Conclusion: The entire data is downloaded from TDS, and after being decoded by Pydap it is not correct.


This was expected, since the TDS correctly serializes first the variables at the root level, and then serializes any hierarchies. Pydap deserializes data inside Groups first, and last all the variables at root level.

